# Audio Processor #
### Spring 2026 ###
### Final Project for Gustave Rohde, TuTh 2:00pm ###
### Created By: ###
* Matthew Bach
* Steve Philipose
* Ishmam 

[Github Repository](https://github.com/Littleamish2/audio_processor_effects.git)


### Architecture: ###
1. Input: audio input (real time stream or recorded file), filter parameters
2. Apply filters to audio input 
3. Output: Filtered audio 

### Effects: ###
* Equalization (Matthew)
* Reverb (Ishmam)
* Delay (Steve)
* Chorus Modulation (Matthew)
* Phaser modulation (Ishmam)
* Flanger Modulation (Steve)

In [ ]:
import math 
from matplotlib import pyplot as plt 
import numpy as np 
from time import time 
import scipy
from scipy.io import wavfile
from IPython.display import display

# Input #

WAV File Processing

In [10]:
## Read WAV File
## Output Sampling Frequency & Mono Signal
def read_wav(fname):
    # Read the wav file given by fname and return the sampling frequency (fs) and
    # the audio data (mono_signal) as a np array
    fs, data = wavfile.read(fname)

    # Convert the data back to floating point values ranging from -1.0 to 1.0
    normalized_data = data / np.iinfo(np.int16).max

    if data.shape[1] > 1: # Covert Stereo to mono
        mono_signal = np.mean(normalized_data, axis=1)

    return fs, mono_signal

Fourier Transform for signal analysis

In [ ]:
def time_to_freq(y, samplerate):
  # Returns the fourier transform of a signal
  # as well as the corresponding frequencies
  n = len(y)
  Y_full = np.fft.fft(y)
  freq_full = np.fft.fftfreq(n, d=1/samplerate)
  return Y_full, freq_full

def freq_to_time(Y):
  # Reconstructs the signal
  y = np.fft.ifft(Y)
  y = np.real(y)
  return y

def plot_freq(Y, freqs):
  # Plot only the magnitude of the FT for only positive frequencies
  # (ignore complex values and negative frequencies)
  plt.figure(figsize=(2.5,2.5))
  inds = freqs >=0
  plt.plot(freqs[inds], np.absolute(Y[inds]))
  plt.xlabel("Frequency")
  plt.ylabel("Fourier Transform Magnitude")

# Effects #

In [ ]:
from enum import Enum, auto

class Curve(Enum):

    FLAT = auto()           ## Flat gain across all frequencies
    SMILEY_FACE = auto()    ## Based on equal-loudness curve 
    FROWNIE_FACE = auto()   ## Opposite of equal-loudness curve
    HIGH_FREQ = auto()      ## Applies only to high frequencies
    LOW_FREQ = auto()       ## Applies only to low frequencies

HIGH_FREQUENCY = 10000
LOW_FREQUENCY = 100


def equalizer(signal, sample_frequency, gain=1.0, curve_type=Curve.FLAT):
    freq_signal, freqs = time_to_freq(signal, sample_frequency)

    match curve_type:
        case Curve.FLAT:
            freq_signal *= gain
            return freq_to_time(freq_signal)
        case Curve.SMILEY_FACE: 
            ## Curve Based On a Cosine-Wave 
            ## Wave with f = f max (one full cycle)
            ## Scoop from gain (on low/high frequencies) to 1 (on mid frequencies)
            
            amplitude = (gain-1)/2
            dc = amplitude + 1
            freq_signal *= dc + amplitude * np.cos(2 * np.pi * freqs / max(freqs)) 

            return freq_to_time(freq_signal)
        
        case Curve.FROWNIE_FACE:
            ## Same as Smiley Face but flip the cosine wave (opposite phase)
            ## Scoop from 1 (on low/high frequencies) to gain (on mid frequencies)

            amplitude = (gain-1)/2 
            dc = amplitude + 1
            freq_signal *= dc + amplitude * -1 * np.cos(2 * np.pi * freqs / max(freqs)) 

            return freq_to_time(freq_signal)
        
        case Curve.HIGH_FREQ:
            return freq_to_time(freq_signal[freqs > HIGH_FREQUENCY] * gain)
        case Curve.LOW_FREQ:
            return freq_to_time(freq_signal[freqs < LOW_FREQUENCY] * gain)

    return signal

In [ ]:
def reverb(signal, sample_frequency):
    # Placeholder for reverb implementation
    return signal

In [ ]:
def delay(signal, sample_frequency):
    # Placeholder for delay implementation
    return signal

In [ ]:
def chorusMod(signal, sample_frequency):
    # Placeholder for chorus modulation implementation
    return signal

In [ ]:
def phaserMod(signal, sample_frequency):
    # Placeholder for phaser modulation implementation
    return signal

In [ ]:
def flangerMod(signal, sample_frequency):
    # Placeholder for flanger modulation implementation
    return signal

# Output #


WAV File Processing

In [11]:
# Write WAV File (fileName, signal, sampling frequency)
def write_wav(fname, sig, fs):
  scaled = np.int16(sig / np.max(np.abs(sig)) * np.iinfo(np.int16).max)
  wavfile.write(fname, fs, scaled)

# Audio Processor #

In [ ]:
## Read Input WAV File
input_file = open("input.wav", "r")
fs, mono_signal = read_wav("input.wav")
input_file.close()

In [ ]:
## Apply Audio Effects
equalized_signal = equalizer(mono_signal, fs)
reverb_signal = reverb(mono_signal, fs)
delay_signal = delay(mono_signal, fs)
chorus_signal = chorusMod(mono_signal, fs)
phaser_signal = phaserMod(mono_signal, fs)
flanger_signal = flangerMod(mono_signal, fs)

all_in_one_signal = flangerMod(chorusMod(phaserMod(delay(reverb(equalizer(mono_signal, fs), fs), fs), fs), fs), fs)

In [ ]:
## Write Output WAV Files
write_wav("equalized_signal.wav", equalized_signal, fs)
write_wav("reverb_signal.wav", reverb_signal, fs)
write_wav("delay_signal.wav", delay_signal, fs)
write_wav("chorus_signal.wav", chorus_signal, fs)
write_wav("phaser_signal.wav", phaser_signal, fs)
write_wav("flanger_signal.wav", flanger_signal, fs)
write_wav("all_in_one_signal.wav", all_in_one_signal, fs)

In [ ]:
## Display Audio for Listening
display(Audio(equalized_signal, rate=fs))
display(Audio(reverb_signal, rate=fs))
display(Audio(delay_signal, rate=fs))
display(Audio(chorus_signal, rate=fs))
display(Audio(phaser_signal, rate=fs))
display(Audio(flanger_signal, rate=fs))
display(Audio(all_in_one_signal, rate=fs))